# Model 4: Meta Prophet Model for Drug `R06`

## Hyperparameter Selection Methodology:
Prophet prior scales evaluated on 2018 Validation Set to optimize trend flexibility.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'R06'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for R06 loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Meta Prophet Hyperparameter Search Code
from prophet import Prophet

prophet_train_df = pd.DataFrame({'ds': train_series.index, 'y': np.log1p(train_series.values)})
prophet_val_df   = pd.DataFrame({'ds': val_series.index})

best_p_cfg = None
best_val_rmsle = float('inf')

print("=== Meta Prophet Grid Search on 2018 Validation Set ===")
for cps in [0.01, 0.05, 0.1]:
    for sps in [0.1, 1.0, 10.0]:
        m = Prophet(growth='linear', yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, changepoint_prior_scale=cps, seasonality_prior_scale=sps)
        m.fit(prophet_train_df)
        forecast_val = m.predict(prophet_val_df)
        pred_val = np.clip(np.expm1(forecast_val['yhat'].values), 0, None)
        met = evaluate_metrics(val_series.values, pred_val)['RMSLE']
        print(f"  * CPS={cps:.2f}, SPS={sps:.1f} : Val RMSLE = {met:.6f}")
        if met < best_val_rmsle:
            best_val_rmsle = met
            best_p_cfg = {'changepoint_prior_scale': cps, 'seasonality_prior_scale': sps}

print(f"Selected Optimal Prophet Configuration: {best_p_cfg}")


=== Meta Prophet Grid Search on 2018 Validation Set ===


21:08:53 - cmdstanpy - INFO - Chain [1] start processing


21:08:54 - cmdstanpy - INFO - Chain [1] done processing


  * CPS=0.01, SPS=0.1 : Val RMSLE = 0.569686


21:08:55 - cmdstanpy - INFO - Chain [1] start processing


...

21:09:03 - cmdstanpy - INFO - Chain [1] done processing


  * CPS=0.10, SPS=1.0 : Val RMSLE = 0.585790


21:09:04 - cmdstanpy - INFO - Chain [1] start processing


21:09:05 - cmdstanpy - INFO - Chain [1] done processing


  * CPS=0.10, SPS=10.0 : Val RMSLE = 0.585501
Selected Optimal Prophet Configuration: {'changepoint_prior_scale': 0.01, 'seasonality_prior_scale': 0.1}


In [3]:
# Step 2: Fit Selected Prophet Config & Forecast 2019 Test
prophet_cb_df = pd.DataFrame({'ds': combined_series.index, 'y': np.log1p(combined_series.values)})
m4_model = Prophet(growth='linear', yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, **best_p_cfg)
m4_model.fit(prophet_cb_df)

future = pd.DataFrame({'ds': test_series.index})
forecast = m4_model.predict(future)
m4_test_pred = np.clip(np.expm1(forecast['yhat'].values), 0, None)

test_metrics = evaluate_metrics(test_series, m4_test_pred)
print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 4: META PROPHET ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_Prophet': m4_test_pred}).to_csv('m4_prophet_preds.csv', index=False)


21:09:06 - cmdstanpy - INFO - Chain [1] start processing


21:09:06 - cmdstanpy - INFO - Chain [1] done processing


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 4: META PROPHET ===
  * RMSLE     : 0.5575
  * RMSE      : 2.5341
  * MAE       : 1.8918
  * WAPE (%)  : 49.5176
